In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [3]:
train_df = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e10/train.csv"
)
test_df = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e10/test.csv"
)

In [4]:
train_df

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,699630,57,612,2,1,1,1,1,3,3,...,2,2,4,0,0.0,Female,Loyal Customer,Business travel,Eco,False
699631,699631,30,862,2,2,2,3,4,2,4,...,4,2,4,24,34.0,Female,disloyal Customer,Business travel,Eco,False
699632,699632,49,3148,5,5,5,5,4,4,4,...,5,5,5,0,0.0,Male,Loyal Customer,Business travel,Business,True
699633,699633,61,679,5,2,2,2,5,4,4,...,5,5,1,0,0.0,Female,Loyal Customer,Business travel,Eco,True


In [5]:
test_df

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
0,699635,25,1197,4,4,4,2,3,4,3,...,5,2,1,3,0,0.0,Female,disloyal Customer,Business travel,Eco
1,699636,36,829,5,5,5,5,5,4,5,...,2,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Eco
2,699637,45,472,4,5,5,5,4,4,4,...,3,1,3,4,0,0.0,Male,Loyal Customer,Business travel,Eco
3,699638,45,762,2,2,2,5,3,2,3,...,4,4,4,3,0,0.0,Female,disloyal Customer,Business travel,Business
4,699639,34,594,1,2,1,1,2,1,2,...,4,2,1,2,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299839,999474,32,1725,3,1,3,3,3,3,3,...,3,3,1,3,0,0.0,Female,Loyal Customer,Business travel,Business
299840,999475,24,1325,3,3,3,4,3,3,3,...,2,3,5,3,0,0.0,Female,disloyal Customer,Business travel,Eco
299841,999476,33,190,4,5,4,3,3,4,3,...,3,2,2,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco
299842,999477,59,678,3,5,3,2,4,3,5,...,4,4,4,5,4,0.0,Male,Loyal Customer,Personal Travel,Eco


## Pre-processing

In [6]:
from sklearn.model_selection import train_test_split

TARGET = "satisfaction"

# Separate predictors and target
X = train_df.drop(columns=[TARGET]).copy()
y = train_df[TARGET].copy()

# Initially exclude the identifier from the predictors
id_columns = [
    column for column in X.columns
    if column.lower() in {"id", "passenger_id", "record_id"}
]

X = X.drop(columns=id_columns)

# Create training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)

print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).round(4))

print("\nValidation target proportions:")
display(y_valid.value_counts(normalize=True).round(4))

Training features: (559708, 21)
Validation features: (139927, 21)

Training target proportions:


satisfaction
False    0.5564
True     0.4436
Name: proportion, dtype: float64


Validation target proportions:


satisfaction
False    0.5564
True     0.4436
Name: proportion, dtype: float64

In [7]:
# imports

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [8]:
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()

numerical_features = [
    column for column in X_train.columns
    if column not in categorical_features
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Numerical features: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


## Implementing Logistic Regression


In [9]:
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

baseline_model.fit(X_train, y_train)

positive_class_index = list(
    baseline_model.named_steps["classifier"].classes_
).index(True)

valid_probabilities = baseline_model.predict_proba(
    X_valid
)[:, positive_class_index]

baseline_auc = roc_auc_score(
    y_valid,
    valid_probabilities
)

print(f"\nBaseline validation ROC-AUC: {baseline_auc:.5f}")


Baseline validation ROC-AUC: 0.92046


## using CatBoost

In [10]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

cat_features = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()

X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()

# CatBoost expects categorical feature values to be strings

for col in cat_features:
    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_valid_cb[col] = (
        X_valid_cb[col]
        .fillna("__MISSING__")
        .astype(str)
    )

print("Categorical columns:", cat_features)
print("Training shape:", X_train_cb.shape)
print("Validation shape:", X_valid_cb.shape)

Categorical columns: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Training shape: (559708, 21)
Validation shape: (139927, 21)


In [11]:
# magic number '972' arises from running the code in another notebook
# prior to this.

catboost_model = CatBoostClassifier(
    iterations=972,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid_cb, y_valid),
    early_stopping_rounds=100,
    verbose=100
)

0:	test: 0.9438315	best: 0.9438315 (0)	total: 424ms	remaining: 6m 51s
100:	test: 0.9550846	best: 0.9550846 (100)	total: 34s	remaining: 4m 53s
200:	test: 0.9564448	best: 0.9564448 (200)	total: 1m 8s	remaining: 4m 22s
300:	test: 0.9569096	best: 0.9569096 (300)	total: 1m 42s	remaining: 3m 48s
400:	test: 0.9571918	best: 0.9571944 (399)	total: 2m 16s	remaining: 3m 14s
500:	test: 0.9573318	best: 0.9573318 (500)	total: 2m 50s	remaining: 2m 40s
600:	test: 0.9574264	best: 0.9574264 (600)	total: 3m 24s	remaining: 2m 6s
700:	test: 0.9575059	best: 0.9575059 (700)	total: 3m 58s	remaining: 1m 32s
800:	test: 0.9575632	best: 0.9575638 (796)	total: 4m 32s	remaining: 58.2s
900:	test: 0.9576221	best: 0.9576255 (898)	total: 5m 6s	remaining: 24.2s
971:	test: 0.9576436	best: 0.9576436 (971)	total: 5m 30s	remaining: 0us

bestTest = 0.9576435564
bestIteration = 971



CatBoostClassifier(allow_writing_files=False, depth=8, eval_metric='AUC', iterations=972, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

## Attempt to add noise to the data in a controlled manner

+ Adding gaussian noise of mean =  0, std = 50. Just to see what happens

In [ ]:
def catboost_prep():
    cat_features = X_train.select_dtypes(
        include=["object", "category", "bool", "string"]
    ).columns.tolist()
    
    X_train_cb = X_train.copy()
    X_valid_cb = X_valid.copy()
    
    # CatBoost expects categorical feature values to be strings
    
    for col in cat_features:
        X_train_cb[col] = (
            X_train_cb[col]
            .fillna("__MISSING__")
            .astype(str)
        )
    
        X_valid_cb[col] = (
            X_valid_cb[col]
            .fillna("__MISSING__")
            .astype(str)
        )
    
    print("Categorical columns:", cat_features)
    print("Training shape:", X_train_cb.shape)
    print("Validation shape:", X_valid_cb.shape)
    return cat_features, X_train_cb, X_valid_cb 

In [ ]:
def log_reg_prep():
    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool", "string"]
    ).columns.tolist()
    
    numerical_features = [
        column for column in X_train.columns
        if column not in categorical_features
    ]
    
    print("Categorical features:", categorical_features)
    print("Numerical features:", numerical_features)
    
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])
    
    preprocessor = ColumnTransformer([
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])
    return preprocessor

In [ ]:
# --- 1. SETUP CORRUPTION FUNCTIONS ---

def get_boundary_indices(model, X_valid_eval, y_valid, preprocessor=None, is_catboost=False):
    """Identifies training rows near the decision boundary using validation/train predictions."""
    # Note: Pass model predictions on X_train to find training points near boundary
    if is_catboost:
        probs = model.predict_proba(X_valid_eval)[:, 1]
    else:
        probs = model.predict_proba(X_valid_eval)[:, 1]
        
    boundary_mask = (probs >= 0.40) & (probs <= 0.60) # Adjustable boundary window
    return np.where(boundary_mask)[0], np.arange(len(X_valid_eval))

def corrupt_single_feature(X, feature_name, corruption_fraction=0.05, target_indices=None):
    """
    Corrupts a single specified feature either weakly or strongly.
    """
    X_corr = X.copy()
    n_samples = int(len(X) * corruption_fraction)
    
    if target_indices is None:
        target_indices = np.arange(len(X))
        
    # Ensure we don't sample more than available indices
    n_samples = min(n_samples, len(target_indices))
    chosen_rows = np.random.choice(target_indices, size=n_samples, replace=False)

    if pd.api.types.is_numeric_dtype(X_corr[feature_name]):
        mean, std = X_corr[feature_name].mean(), X_corr[feature_name].std()
        # Extreme noise injection
        noise_vals = np.random.normal(mean, std * 4.0, size=len(chosen_rows))
        X_corr.iloc[chosen_rows, X_corr.columns.get_loc(feature_name)] = noise_vals
    else:
        unique_vals = X_corr[feature_name].dropna().unique()
        rand_vals = np.random.choice(unique_vals, size=len(chosen_rows))
        X_corr.iloc[chosen_rows, X_corr.columns.get_loc(feature_name)] = rand_vals
            
    return X_corr, chosen_rows

In [ ]:
def weak_corrupt_single_feature(X, feature_name, corruption_fraction=0.05, target_indices=None):
    """
    Corrupts a single specified feature weakly.
    """
    X_corr = X.copy()
    n_samples = int(len(X) * corruption_fraction)
    
    if target_indices is None:
        target_indices = np.arange(len(X))
        
    # Ensure we don't sample more than available indices
    n_samples = min(n_samples, len(target_indices))
    chosen_rows = np.random.choice(target_indices, size=n_samples, replace=False)

    # Replace valid rating/category with another valid one
    valid_vals = X_corr[feature_name].dropna().unique()
    if len(valid_vals) > 1:
        current_vals = X_corr.iloc[chosen_rows][feature_name].values
        new_vals = np.array([np.random.choice([r for r in valid_vals if r != val]) for val in current_vals])
        X_corr.iloc[chosen_rows, X_corr.columns.get_loc(feature_name)] = new_vals
        
    return X_corr, chosen_rows

In [ ]:
# --- 2. EVALUATION & SWEEP PIPELINE ---

def evaluate_logistic_regression(preprocessor, X_train_curr, y_train, X_valid, y_valid):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ])
    model.fit(X_train_curr, y_train)
    pos_idx = list(model.named_steps["classifier"].classes_).index(True)
    probs = model.predict_proba(X_valid)[:, pos_idx]
    return roc_auc_score(y_valid, probs), model

def evaluate_catboost(X_train_cb_curr, y_train, X_valid_cb, y_valid, cat_features):
    model = CatBoostClassifier(
        iterations=500, # Can use 972 as, in the interests of time this has been done
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    )
    model.fit(
        X_train_cb_curr, y_train,
        cat_features=cat_features,
        eval_set=(X_valid_cb, y_valid),
        early_stopping_rounds=50,
        verbose=False
    )
    probs = model.predict_proba(X_valid_cb)[:, 1]
    return roc_auc_score(y_valid, probs), model

In [ ]:
# --- 3. RUNNING THE SWEEP ACROSS ALL FEATURES ---

def run_feature_robustness_sweep(X_train, y_train, X_valid, y_valid, X_train_cb, X_valid_cb, cat_features, preprocessor):
    results = []
    
    # First, get baseline models to find boundary indices
    print("Training baseline models to locate decision boundaries...")
    _, base_lr_model = evaluate_logistic_regression(preprocessor, X_train, y_train, X_valid, y_valid)

    '''
    Find boundary indices using Logistic Regression train/valid predictions (or CatBoost)
    train_probs_lr = base_lr_model.predict_proba(X_train)[:, list(base_lr_model.named_steps["classifier"].classes_).index(True)]
    boundary_indices = np.where((train_probs_lr >= 0.40) & (train_probs_lr <= 0.60))[0]
    uniform_indices = np.arange(len(X_train))
    '''
    
    features_to_test = [col for col in X_train.columns if col != 'id']
    
    for feature in features_to_test:
        print(f"\n--- Testing Feature: {feature} ---")
        
        '''for mode_name, indices in [('Uniform', uniform_indices), ('Boundary', boundary_indices)]:
        ''' 
        # Apply paired corruption to standard train set (for LR) and CB train set simultaneously
        X_train_corr, mask = corrupt_single_feature(
            X_train, feature, corruption_fraction=0.40, target_indices=None
        )
        
        # If using separate formatting for CatBoost dataframe, apply the exact same row modifications:
        X_train_cb_corr = X_train_cb.copy()
        X_train_cb_corr.iloc[mask] = X_train_corr.iloc[mask]
        
        # Evaluate Logistic Regression
        lr_auc, _ = evaluate_logistic_regression(preprocessor, X_train_corr, y_train, X_valid, y_valid)

        print(f'LR_AUC: {lr_auc}')
        
        # Evaluate CatBoost
        cb_auc, _ = evaluate_catboost(X_train_cb_corr, y_train, X_valid_cb, y_valid, cat_features)

        print(f'nCatBoost_AUC: {cb_auc}')
        
        results.append({
            'Feature': feature,
            #'Corruption_Type': corr_type,
            #'Strategy': mode_name,
            'LR_AUC': lr_auc,
            'CatBoost_AUC': cb_auc
        })
            
    return pd.DataFrame(results)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

cat_features, X_train_cb, X_valid_cb = catboost_prep()
preprocessor = log_reg_prep()
run_feature_robustness_sweep(X_train, y_train, X_valid, y_valid, X_train_cb, X_valid_cb, cat_features, preprocessor)